# Americas Transportation Analytics
## Notebook 01 — Exploración y diagnóstico de calidad de datos

**Autor:** Sergio Corona
**Fecha:** julio 2026

---

### Contexto de negocio

Una manufacturera global opera una red de transporte en Américas (US–México–Canadá)
con seis modos: TL, LTL, Parcel, Air, Ocean y Drayage. El área de transporte necesita
identificar oportunidades de ahorro y evaluar el desempeño de sus transportistas,
pero la extracción del TMS presenta problemas de calidad que impiden confiar en los KPI.

### Objetivo de este notebook

Diagnosticar el estado del archivo `fact_shipments_raw.csv` antes de cualquier análisis:
cuantificar nulos, duplicados, inconsistencias de formato y valores imposibles, para
sustentar las decisiones del pipeline de limpieza.

### Nota sobre los datos

Dataset **sintético**, generado para replicar la estructura de un TMS real.
No corresponde a información de ninguna empresa.

## 1. Carga de datos

Se carga la extracción cruda del TMS. Se usa `low_memory=False` para que pandas
lea el archivo completo antes de inferir tipos y evite advertencias de tipos mixtos.

In [1]:
import pandas as pd
import numpy as np

# Mostrar todas las columnas al imprimir (por defecto pandas las trunca)
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 200)

# Ruta relativa: el notebook vive en /notebooks, los datos en /data/raw
ruta = '../data/raw/fact_shipments_raw.csv'
raw = pd.read_csv(ruta, low_memory=False)

print(f'Filas:    {raw.shape[0]:,}')
print(f'Columnas: {raw.shape[1]}')

Filas:    221,320
Columnas: 39


**Resultado:** 221,320 filas × 39 columnas.

La documentación del sistema indica que el universo real es de 220,000 embarques,
lo que anticipa aproximadamente **1,320 registros duplicados** por validar.

## 2. Estructura y tipos de dato

Se revisa el esquema para identificar dos cosas: columnas con valores faltantes
y columnas cuyo tipo inferido no corresponde a su naturaleza.

In [2]:
raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 221320 entries, 0 to 221319
Data columns (total 39 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   shipment_id            221320 non-null  str    
 1   tms_reference          221320 non-null  str    
 2   ship_date              221320 non-null  str    
 3   mode                   221320 non-null  str    
 4   service_level          221320 non-null  str    
 5   carrier_id             220433 non-null  str    
 6   origin_site_id         221320 non-null  str    
 7   dest_site_id           221320 non-null  str    
 8   origin_country         221320 non-null  str    
 9   dest_country           221320 non-null  str    
 10  lane_id                221320 non-null  str    
 11  cross_border_flag      221320 non-null  int64  
 12  incoterm               214680 non-null  str    
 13  business_unit          221320 non-null  str    
 14  customer_segment       221320 non-null  str    

**Hallazgo crítico:** las cinco columnas temporales (`ship_date`, `planned_pickup_ts`,
`actual_pickup_ts`, `planned_delivery_ts`, `actual_delivery_ts`) fueron inferidas
como texto (`str`), no como fechas.

**Implicación:** en su estado actual es imposible calcular días de tránsito,
detectar entregas fuera de secuencia o agrupar por periodo. La conversión de tipos
es prerrequisito de todo el análisis de servicio.

## 3. Análisis de completitud

Se cuantifican los valores faltantes por columna, en términos absolutos y relativos.

In [3]:
# isna() marca vacíos; sum() los cuenta, mean() da la proporción
nulos = pd.DataFrame({
    'nulos': raw.isna().sum(),
    'pct': (100 * raw.isna().mean()).round(2)
})

# Solo interesan las columnas que sí tienen faltantes
nulos[nulos['nulos'] > 0].sort_values('nulos', ascending=False)

,nulos,pct
incoterm,6640,3.0
volume_cbm,4422,2.0
accessorial_usd,3318,1.5
weight_kg,2648,1.2
actual_delivery_ts,1767,0.8
carrier_id,887,0.4


**Resultado:** seis columnas con faltantes, ninguna por encima del 3%.

| Columna | Nulos | % | Criticidad |
|---|---|---|---|
| `incoterm` | 6,640 | 3.0% | Baja — nulo puede ser legítimo en tráfico doméstico |
| `volume_cbm` | 4,422 | 2.0% | Media — afecta cálculo de peso volumétrico |
| `accessorial_usd` | 3,318 | 1.5% | Media — ausencia probablemente equivale a cero |
| `weight_kg` | 2,648 | 1.2% | Alta — es el denominador del costo por kg |
| `actual_delivery_ts` | 1,767 | 0.8% | Alta — sin ella no hay medición de OTD |
| `carrier_id` | 887 | 0.4% | **Crítica** — sin transportista el registro no sirve para el scorecard |

**Criterio de evaluación:** el porcentaje por sí solo no determina la gravedad.
`carrier_id` tiene el menor porcentaje pero la mayor criticidad, porque es la llave
del análisis de desempeño por transportista. La decisión de tratamiento debe
responder al uso analítico, no al volumen de faltantes.

## 4. Detección de duplicados

Se evalúan dos escenarios que exigen tratamientos distintos:

- **Duplicado exacto:** fila idéntica en las 39 columnas. Es ruido de extracción
  y puede eliminarse sin pérdida de información.
- **`shipment_id` repetido con datos distintos:** el mismo embarque cargado con
  información diferente. Requiere decidir qué versión conservar.

In [4]:
dup_exactos = raw.duplicated().sum()
dup_id = raw['shipment_id'].duplicated().sum()

print(f'Duplicados exactos (fila completa): {dup_exactos:,}')
print(f'shipment_id repetidos:              {dup_id:,}')
print(f'shipment_id únicos:                 {raw["shipment_id"].nunique():,}')

Duplicados exactos (fila completa): 1,186
shipment_id repetidos:              1,320
shipment_id únicos:                 220,000


**Resultado:**

- Duplicados exactos: **1,186**
- `shipment_id` repetidos: **1,320**
- `shipment_id` únicos: **220,000** ← coincide con el universo documentado

**Hallazgo:** la diferencia entre 1,320 y 1,186 revela **134 registros con el mismo
`shipment_id` pero contenido distinto**. No son duplicados limpios: representan el
mismo embarque cargado con información divergente.

**Implicación:** eliminar duplicados de forma automática (`drop_duplicates()`)
resolvería 1,186 casos pero tomaría una decisión arbitraria sobre los 134 restantes.
Se requiere inspección previa para definir un criterio de conservación defendible.

In [5]:
# IDs que aparecen más de una vez
ids_repetidos = raw['shipment_id'][raw['shipment_id'].duplicated()].unique()

# Filas de esos IDs, quitando las que son duplicado exacto
conflictivos = raw[raw['shipment_id'].isin(ids_repetidos)].drop_duplicates()

# Nos quedan solo los IDs que siguen repitiéndose = los que tienen datos divergentes
ids_conflicto = conflictivos['shipment_id'].value_counts()
ids_conflicto = ids_conflicto[ids_conflicto > 1].index

print(f'IDs con versiones divergentes: {len(ids_conflicto)}')

# Inspeccionamos los primeros 3 casos
raw[raw['shipment_id'].isin(ids_conflicto[:3])].sort_values('shipment_id')

IDs con versiones divergentes: 134


,shipment_id,tms_reference,ship_date,mode,service_level,carrier_id,origin_site_id,dest_site_id,origin_country,dest_country,lane_id,cross_border_flag,incoterm,business_unit,customer_segment,weight_kg,volume_cbm,chargeable_weight_kg,pallets,distance_km,linehaul_cost_usd,fuel_surcharge_usd,accessorial_usd,customs_fee_usd,detention_usd,total_cost_usd,invoice_amount_usd,invoice_variance_usd,planned_transit_days,actual_transit_days,planned_pickup_ts,actual_pickup_ts,planned_delivery_ts,actual_delivery_ts,on_time_pickup_flag,on_time_delivery_flag,tender_accepted_flag,damaged_flag,claim_flag
1337,SHP1013457,TMS-69399450,2023-05-29,TL,Standard,C003,MXTIJ,MXMTY,MX,MX,MXTIJ-MXMTY,0,DAP,Industrial,Contract Mfg,19200.1,97.708,19200.1,70,2273.0,4905.45,1003.65,0.0,0.0,0.0,5909.10,5935.34,26.24,5.0,5.26,2023-05-29,2023-05-29 00:00:00,2023-06-03,2023-06-03 06:14:24.000000000,1,1,1,0,0
97661,SHP1013457,TMS-69399450,2023-05-29,TL,Standard,C003,MXTIJ,MXMTY,Mexico,MX,MXTIJ-MXMTY,0,DAP,Industrial,Contract Mfg,19200.1,97.708,19200.1,70,2273.0,4905.45,1003.65,0.0,0.0,0.0,5909.10,5935.34,26.24,5.0,5.26,2023-05-29,2023-05-29 00:00:00,2023-06-03,2023-06-03 06:14:24.000000000,1,1,1,0,0
2456,SHP1077486,TMS-54358651,2025-09-23,LTL,Expedited,C009,CATOR,CAMTL,CA,CA,CATOR-CAMTL,0,DAP,Automotive,Aftermarket,734.7,3.015,734.7,3,667.4,174.00,29.79,0.0,0.0,0.0,203.79,200.09,-3.70,3.0,1.52,2025-09-23,2025-09-23 14:36:00,2025-09-26,2025-09-25 03:04:48.000000000,0,1,1,0,0
39342,SHP1077486,TMS-54358651,2025-09-23,LTL,Expedited,C009,CATOR,CAMTL,Canada,CA,CATOR-CAMTL,0,DAP,Automotive,Aftermarket,734.7,3.015,734.7,3,667.4,174.00,29.79,0.0,0.0,0.0,203.79,200.09,-3.70,3.0,1.52,2025-09-23,2025-09-23 14:36:00,2025-09-26,2025-09-25 03:04:48.000000000,0,1,1,0,0
1827,SHP1200334,TMS-20266542,2025-06-09,Truckload,Standard,C003,USFRE,USHAR,US,US,USFRE-USHAR,0,DAP,Automotive,Aftermarket,8854.7,42.932,8854.7,31,4886.4,5438.85,916.45,113.1,0.0,0.0,6468.40,6045.52,-422.88,8.0,6.92,2025-06-09,2025-06-09 00:00:00,2025-06-17,2025-06-15 22:04:48.000000000,1,1,1,0,0
38529,SHP1200334,TMS-20266542,2025-06-09,TL,Standard,C003,USFRE,USHAR,US,US,USFRE-USHAR,0,DAP,Automotive,Aftermarket,8854.7,42.932,8854.7,31,4886.4,5438.85,916.45,113.1,0.0,0.0,6468.40,6045.52,-422.88,8.0,6.92,2025-06-09,2025-06-09 00:00:00,2025-06-17,2025-06-15 22:04:48.000000000,1,1,1,0,0


**Inspección de los 134 casos divergentes:**

Al comparar las versiones de tres IDs afectados, las diferencias se concentran
exclusivamente en columnas categóricas de formato:

| shipment_id | Columna | Versión A | Versión B |
|---|---|---|---|
| SHP1013457 | `origin_country` | `MX` | `Mexico` |
| SHP1077486 | `origin_country` | `CA` | `Canada` |
| SHP1200334 | `mode` | `TL` | `Truckload` |

El resto de los campos —peso, volumen, costos, fechas, flags de servicio— es
**idéntico** entre ambas versiones.

**Conclusión:** no se trata de registros en conflicto, sino de duplicados
enmascarados por inconsistencia de formato.

**Decisión de diseño del pipeline:** la normalización de categóricas debe ejecutarse
**antes** de la deduplicación. El orden inverso —el más intuitivo— dejaría 134
duplicados sin detectar, inflando el conteo de embarques y el gasto total,
y contaminando todos los KPI derivados.

In [6]:
# Copia de trabajo con las categóricas normalizadas
prueba = raw.copy()

prueba['mode'] = prueba['mode'].str.strip().str.upper()
for col in ['origin_country', 'dest_country']:
    prueba[col] = prueba[col].str.strip().str.upper().replace({
        'USA': 'US', 'MEXICO': 'MX', 'CANADA': 'CA'
    })

print(f'Duplicados exactos ANTES de normalizar:   {raw.duplicated().sum():,}')
print(f'Duplicados exactos DESPUÉS de normalizar: {prueba.duplicated().sum():,}')
print(f'Duplicados recuperados:                   {prueba.duplicated().sum() - raw.duplicated().sum():,}')

Duplicados exactos ANTES de normalizar:   1,186
Duplicados exactos DESPUÉS de normalizar: 1,290
Duplicados recuperados:                   104


## 5. Inventario de valores categóricos

Antes de escribir reglas de normalización se levanta el inventario real de valores.
Asumir cuáles existen es la principal fuente de reglas incompletas.

**`mode` — 14 valores distintos para 6 modos reales:**

| Canónico | Variantes encontradas | Registros afectados |
|---|---|---|
| Parcel | `PARCEL`, `parcel` | 3,811 |
| LTL | `` ltl`` (espacio inicial) | 745 |
| TL | `Truckload` | 639 |
| Ocean | `OCEAN`, `OCEAN FCL` | 824 |
| Air | `AIR` | 432 |
| Drayage | `DRAYAGE` | 247 |

Total con formato inconsistente: **6,698 registros (3.0%)**

**`origin_country` — 6 valores para 3 países:** `USA`→US (2,346), `Mexico`→MX (1,703),
`Canada`→CA (377). Total: **4,426 registros (2.0%)**

**`dest_country` — ya normalizado.** Solo contiene US, MX y CA.

In [7]:
# Inventario completo de valores en las columnas categóricas problemáticas
print('=== MODE ===')
print(raw['mode'].value_counts(dropna=False))

print('\n=== ORIGIN_COUNTRY ===')
print(raw['origin_country'].value_counts(dropna=False))

print('\n=== DEST_COUNTRY ===')
print(raw['dest_country'].value_counts(dropna=False))

=== MODE ===
mode
Parcel       80898
LTL          52269
TL           41283
Ocean        16704
Air          14901
Drayage       8567
PARCEL        2552
parcel        1259
 ltl           745
Truckload      639
OCEAN          546
AIR            432
OCEAN FCL      278
DRAYAGE        247
Name: count, dtype: int64

=== ORIGIN_COUNTRY ===
origin_country
US        114415
MX         83975
CA         18504
USA         2346
Mexico      1703
Canada       377
Name: count, dtype: int64

=== DEST_COUNTRY ===
dest_country
US    111820
MX     72646
CA     36854
Name: count, dtype: int64


In [8]:
# Diccionario de mapeo: la llave es el valor ya en mayúsculas y sin espacios
MAPA_MODO = {
    'TL': 'TL',        'TRUCKLOAD': 'TL',
    'LTL': 'LTL',
    'PARCEL': 'Parcel',
    'AIR': 'Air',
    'OCEAN': 'Ocean',  'OCEAN FCL': 'Ocean',
    'DRAYAGE': 'Drayage',
}

MAPA_PAIS = {
    'US': 'US', 'USA': 'US',
    'MX': 'MX', 'MEXICO': 'MX',
    'CA': 'CA', 'CANADA': 'CA',
}

prueba = raw.copy()

# strip() quita espacios, upper() unifica mayúsculas, map() traduce sinónimos
prueba['mode'] = prueba['mode'].str.strip().str.upper().map(MAPA_MODO)

for col in ['origin_country', 'dest_country']:
    prueba[col] = prueba[col].str.strip().str.upper().map(MAPA_PAIS)

# Verificación: si algún valor no estaba en el diccionario, map() lo deja como nulo
print('Valores sin mapear:')
print(f'  mode:           {prueba["mode"].isna().sum()}')
print(f'  origin_country: {prueba["origin_country"].isna().sum()}')
print(f'  dest_country:   {prueba["dest_country"].isna().sum()}')

print(f'\nValores únicos finales en mode: {sorted(prueba["mode"].unique())}')

print(f'\nDuplicados exactos ANTES:   {raw.duplicated().sum():,}')
print(f'Duplicados exactos DESPUÉS: {prueba.duplicated().sum():,}')
print(f'Recuperados:                {prueba.duplicated().sum() - raw.duplicated().sum():,}')

Valores sin mapear:
  mode:           0
  origin_country: 0
  dest_country:   0

Valores únicos finales en mode: ['Air', 'Drayage', 'LTL', 'Ocean', 'Parcel', 'TL']

Duplicados exactos ANTES:   1,186
Duplicados exactos DESPUÉS: 1,309
Recuperados:                123


In [9]:
# IDs que siguen repetidos incluso después de normalizar y quitar duplicados exactos
restantes = prueba.drop_duplicates()
conteo = restantes['shipment_id'].value_counts()
ids_pendientes = conteo[conteo > 1].index.tolist()

print(f'IDs con divergencia pendiente: {len(ids_pendientes)}')

# Para cada ID, detectar en qué columnas difieren sus versiones
columnas_divergentes = {}

for sid in ids_pendientes:
    bloque = restantes[restantes['shipment_id'] == sid]
    for col in bloque.columns:
        # nunique con dropna=False cuenta el nulo como un valor más
        if bloque[col].nunique(dropna=False) > 1:
            columnas_divergentes[col] = columnas_divergentes.get(col, 0) + 1

print('\nColumnas donde difieren las versiones:')
for col, n in sorted(columnas_divergentes.items(), key=lambda x: -x[1]):
    print(f'  {col:25s} {n} casos')

IDs con divergencia pendiente: 11

Columnas donde difieren las versiones:
  total_cost_usd            7 casos
  actual_delivery_ts        2 casos
  weight_kg                 2 casos


In [10]:
# Inspección enfocada: solo las columnas relevantes
cols_ver = ['shipment_id', 'mode', 'weight_kg', 'total_cost_usd',
            'linehaul_cost_usd', 'actual_pickup_ts', 'actual_delivery_ts']

restantes[restantes['shipment_id'].isin(ids_pendientes)][cols_ver].sort_values('shipment_id')

,shipment_id,mode,weight_kg,total_cost_usd,linehaul_cost_usd,actual_pickup_ts,actual_delivery_ts
60827,SHP1058118,TL,10675.0,-8274.06,6334.94,2025-07-14 00:30:00,2025-07-20 16:20:24.000000000
135775,SHP1058118,TL,10675.0,8274.06,6334.94,2025-07-14 00:30:00,2025-07-20 16:20:24.000000000
67153,SHP1065994,Parcel,3.3,384.53,18.25,2024-05-13 01:42:00,2024-05-16 01:56:24.000000000
109851,SHP1065994,Parcel,3.3,-384.53,18.25,2024-05-13 01:42:00,2024-05-16 01:56:24.000000000
126734,SHP1078959,Parcel,15.6,235.98,121.88,2023-11-23 00:00:00,2023-11-29 02:38:24.000000000
196601,SHP1078959,Parcel,15.6,235.98,121.88,2023-11-23 00:00:00,2023-11-21 00:00:00.000000000
15119,SHP1101787,Drayage,14298.2,4201.29,3390.18,2025-08-15 01:00:00,2025-08-22 02:40:48.000000000
205604,SHP1101787,Drayage,14298.2,-4201.29,3390.18,2025-08-15 01:00:00,2025-08-22 02:40:48.000000000
123030,SHP1111694,TL,14311300.0,2541.06,2044.62,2023-07-20 00:00:00,2023-07-22 09:07:12.000000000
129114,SHP1111694,TL,14311.3,2541.06,2044.62,2023-07-20 00:00:00,2023-07-22 09:07:12.000000000


## 6. Resolución de duplicados con divergencia

Tras normalizar categóricas, 11 IDs conservan versiones divergentes en tres columnas:
`total_cost_usd` (7 casos), `weight_kg` (2) y `actual_delivery_ts` (2).

**Patrón identificado:** en los 11 casos una copia contiene el valor válido y la otra
una versión corrupta:

| Tipo | Casos | Patrón |
|---|---|---|
| Signo invertido | 7 | `total_cost_usd` × (−1); los componentes de costo permanecen positivos |
| Error de unidad | 2 | `weight_kg` × 1000 (gramos registrados como kilogramos) |
| Secuencia imposible | 2 | `actual_delivery_ts` anterior a `actual_pickup_ts` |

**Valor analítico de estos casos:** el archivo contiene aproximadamente 400 costos
negativos y 250 pesos inflados sin registro gemelo. Estos 11 duplicados funcionan como
**muestra de validación natural**: confirman empíricamente que la corrupción responde a
transformaciones deterministas (×−1 y ×1000), lo que permite justificar las reglas de
corrección aplicadas al resto del universo con evidencia y no con supuestos.

**Criterio de conservación:** ante versiones divergentes se conserva la que satisface
las reglas de validez de negocio —costo positivo, peso dentro de rango plausible y
entrega posterior al pickup— en lugar de aplicar un criterio arbitrario como `keep='first'`.

In [11]:
dedup = prueba.copy()

# Puntaje de validez: cuántas reglas de negocio cumple cada fila
dedup['_valido'] = (
    (dedup['total_cost_usd'] > 0).astype(int) +
    (dedup['weight_kg'] < 100_000).astype(int) +
    (pd.to_datetime(dedup['actual_delivery_ts'], errors='coerce') >=
     pd.to_datetime(dedup['actual_pickup_ts'], errors='coerce')).astype(int)
)

# Ordenar de mayor a menor puntaje y conservar la mejor versión de cada ID
dedup = (dedup
         .sort_values('_valido', ascending=False)
         .drop_duplicates('shipment_id', keep='first')
         .drop(columns='_valido')
         .sort_index())

print(f'Filas iniciales: {len(raw):,}')
print(f'Filas tras deduplicar: {len(dedup):,}')
print(f'Eliminadas: {len(raw) - len(dedup):,}')
print(f'\n¿IDs únicos? {dedup["shipment_id"].is_unique}')

Filas iniciales: 221,320
Filas tras deduplicar: 220,000
Eliminadas: 1,320

¿IDs únicos? True


**Resultado de la deduplicación:**

| Métrica | Valor |
|---|---|
| Filas iniciales | 221,320 |
| Filas finales | 220,000 |
| Registros eliminados | 1,320 |
| Unicidad de `shipment_id` | ✔ |

El resultado coincide exactamente con el universo documentado de 220,000 embarques.

**Secuencia validada del pipeline:**

1. Normalizar categóricas → expone 134 duplicados enmascarados por formato
2. Deduplicar con criterio de validez de negocio → resuelve 11 casos divergentes
3. Corregir valores imposibles → pendiente
4. Convertir tipos temporales → pendiente
5. Tratar nulos → pendiente

Invertir los pasos 1 y 2 —el orden intuitivo— habría dejado 134 duplicados sin detectar,
inflando el conteo de embarques y el gasto total reportado.

## 7. Detección de valores imposibles

### 7.1 Costos negativos

393 registros con `total_cost_usd` negativo. Ningún componente del costo
(`linehaul`, `fuel_surcharge`, `accessorial`, `customs`, `detention`) presenta
valores negativos.

**Implicación:** la corrupción afecta únicamente a la columna agregada, lo que permite
reconstruir el valor correcto sumando componentes y validarlo contra la inversión de signo.

### 7.2 Pesos inflados por error de unidad

**Primer intento — umbral global (mediana × 100 = 45,080 kg): 159 registros detectados.**

Resultado insuficiente. El perfil por modo revela por qué:

| Modo | Mediana | P75 | Máximo |
|---|---|---|---|
| Parcel | 11.1 | 20.3 | 68,000 |
| LTL | 734.6 | 1,265.6 | 4,188,900 |
| TL | 10,953.4 | 13,867.5 | 20,353,100 |
| Ocean | 14,688.6 | 20,726.7 | 26,000,000 |

Un envío Parcel de 3 kg inflado ×1000 resulta en 3,000 kg: absurdo para paquetería,
pero muy por debajo del umbral global. Un criterio único es inaplicable cuando las
escalas difieren en tres órdenes de magnitud.

**Segundo intento — criterio de ida y vuelta por modo: 242 registros detectados.**

Se marca un valor como inflado solo si cumple ambas condiciones:
1. Excede el percentil 99 de su propio modo
2. Al dividirse entre 1000 regresa al rango plausible de ese modo

La segunda condición es la que aporta solidez: no se declara que el valor es atípico,
se demuestra que la hipótesis del error de unidad lo explica.

| Estadístico | Antes | Corregido |
|---|---|---|
| Mínimo | 1,600 kg | 1.6 kg |
| Mediana | 483,900 kg | 483.9 kg |
| Máximo | 26,000,000 kg | 26,000 kg |

La mediana corregida (483.9 kg) es consistente con la mediana del universo sano (450.8 kg).

**Nota sobre cobertura:** la documentación estima ~250 casos; el método detecta 242.
Los restantes corresponden a registros cuyo valor inflado cae dentro del rango legítimo
de su modo, resultando estadísticamente indistinguibles de valores reales. Se documenta
esta limitación en lugar de relajar el criterio hasta forzar la cifra esperada.

### 7.3 Distancia

`distance_km` no presenta anomalías: rango de 15 a 17,050 km, coherente con
rutas domésticas e intercontinentales. No requiere tratamiento.

In [12]:
# Columnas de costo a auditar
cols_costo = ['linehaul_cost_usd', 'fuel_surcharge_usd', 'accessorial_usd',
              'customs_fee_usd', 'detention_usd', 'total_cost_usd', 'invoice_amount_usd']

print('=== COSTOS NEGATIVOS ===')
for col in cols_costo:
    n = (dedup[col] < 0).sum()
    if n > 0:
        print(f'  {col:22s} {n:>5,} registros')

print('\n=== PESO: distribución ===')
print(dedup['weight_kg'].describe().round(2))

print('\n=== DISTANCIA: distribución ===')
print(dedup['distance_km'].describe().round(2))

=== COSTOS NEGATIVOS ===
  total_cost_usd           393 registros

=== PESO: distribución ===
count      217360.00
mean         9294.66
std        275807.45
min             0.50
25%            15.70
50%           450.80
75%          8558.82
max      26000000.00
Name: weight_kg, dtype: float64

=== DISTANCIA: distribución ===
count    220000.00
mean       2834.48
std        2027.96
min          15.00
25%        1445.80
50%        2621.40
75%        3672.90
max       17049.70
Name: distance_km, dtype: float64


In [13]:
mediana_peso = dedup['weight_kg'].median()
umbral = mediana_peso * 100

sospechosos = dedup[dedup['weight_kg'] > umbral]

print(f'Mediana de peso:        {mediana_peso:,.1f} kg')
print(f'Umbral (mediana × 100): {umbral:,.1f} kg')
print(f'Registros por encima:   {len(sospechosos):,}')

print('\n=== ¿Qué pasa si los dividimos entre 1000? ===')
print((sospechosos['weight_kg'] / 1000).describe().round(2))

print('\n=== Distribución por modo ===')
print(sospechosos['mode'].value_counts())

Mediana de peso:        450.8 kg
Umbral (mediana × 100): 45,080.0 kg
Registros por encima:   159

=== ¿Qué pasa si los dividimos entre 1000? ===
count      159.00
mean      6894.68
std       7539.77
min         48.80
25%        539.85
50%       1831.50
75%      12805.10
max      26000.00
Name: weight_kg, dtype: float64

=== Distribución por modo ===
mode
LTL        69
TL         46
Ocean      18
Air        13
Drayage     8
Parcel      5
Name: count, dtype: int64


In [14]:
# Perfil de peso por modo, usando estadísticos robustos
perfil = dedup.groupby('mode')['weight_kg'].agg(
    n='size',
    mediana='median',
    p25=lambda x: x.quantile(0.25),
    p75=lambda x: x.quantile(0.75),
    maximo='max'
).round(1)

perfil['umbral_x100'] = (perfil['mediana'] * 100).round(0)

print(perfil)

             n  mediana      p25      p75      maximo  umbral_x100
mode                                                              
Air      15248    220.0    113.0    434.4   1055800.0      22000.0
Drayage   8749  13277.3  10897.0  16284.7  22000000.0    1327730.0
LTL      52731    734.6    427.7   1265.6   4188900.0      73460.0
Ocean    17431  14688.6  10529.0  20726.7  26000000.0    1468860.0
Parcel   84202     11.1      6.0     20.3     68000.0       1110.0
TL       41639  10953.4   8646.7  13867.5  20353100.0    1095340.0


In [15]:
# Percentiles de referencia por modo (excluyendo nulos)
p01 = dedup.groupby('mode')['weight_kg'].quantile(0.01)
p99 = dedup.groupby('mode')['weight_kg'].quantile(0.99)

# Mapear el umbral de cada fila según su modo
lim_sup = dedup['mode'].map(p99)
lim_inf = dedup['mode'].map(p01)

# Condición 1: excede el percentil 99 de su modo
excede = dedup['weight_kg'] > lim_sup

# Condición 2: al dividir entre 1000 vuelve al rango plausible del modo
vuelve = (dedup['weight_kg'] / 1000).between(lim_inf, lim_sup)

inflados = excede & vuelve

print(f'Registros con peso inflado ×1000: {inflados.sum():,}')
print('\nPor modo:')
print(dedup.loc[inflados, 'mode'].value_counts())

print('\n=== Comparación antes / después de corregir ===')
comp = pd.DataFrame({
    'actual': dedup.loc[inflados, 'weight_kg'].describe().round(1),
    'corregido': (dedup.loc[inflados, 'weight_kg'] / 1000).describe().round(1)
})
print(comp)

Registros con peso inflado ×1000: 242

Por modo:
mode
Parcel     89
LTL        69
TL         45
Ocean      18
Air        13
Drayage     8
Name: count, dtype: int64

=== Comparación antes / después de corregir ===
           actual  corregido
count       242.0      242.0
mean    4518450.0     4518.4
std     6932867.4     6932.9
min        1600.0        1.6
25%       16675.0       16.7
50%      483900.0      483.9
75%     9167800.0     9167.8
max    26000000.0    26000.0


In [16]:
dedup.to_pickle('../data/processed/01_deduplicado.pkl')
print(f'Guardado: {len(dedup):,} filas')

Guardado: 220,000 filas


## 8. Diagnóstico temporal

Las cinco columnas de fecha fueron inferidas como texto en la carga. Su conversión
es prerrequisito de todo el análisis de servicio: nivel de cumplimiento (OTD/OTP),
días de tránsito reales y detección de cuellos de botella.

Se evalúan tres aspectos:
1. **Parseabilidad** — ¿todos los valores tienen formato de fecha válido?
2. **Cobertura** — ¿el rango temporal corresponde al periodo documentado?
3. **Coherencia de secuencia** — ¿el orden de los hitos respeta la lógica operativa?

In [17]:
cols_fecha = ['ship_date', 'planned_pickup_ts', 'actual_pickup_ts',
              'planned_delivery_ts', 'actual_delivery_ts']

fechas = dedup.copy()

print('=== CONVERSIÓN ===')
for col in cols_fecha:
    antes = fechas[col].notna().sum()
    # errors='coerce' convierte lo no parseable en NaT en vez de detener el proceso
    fechas[col] = pd.to_datetime(fechas[col], errors='coerce')
    despues = fechas[col].notna().sum()
    print(f'  {col:22s} con valor: {despues:>7,}  no parseables: {antes - despues:>5,}')

print('\n=== RANGO TEMPORAL ===')
for col in cols_fecha:
    print(f'  {col:22s} {fechas[col].min()}  →  {fechas[col].max()}')

=== CONVERSIÓN ===
  ship_date              con valor: 220,000  no parseables:     0
  planned_pickup_ts      con valor: 220,000  no parseables:     0
  actual_pickup_ts       con valor: 220,000  no parseables:     0
  planned_delivery_ts    con valor: 220,000  no parseables:     0
  actual_delivery_ts     con valor: 218,244  no parseables:     0

=== RANGO TEMPORAL ===
  ship_date              2023-01-01 00:00:00  →  2026-06-30 00:00:00
  planned_pickup_ts      2023-01-01 00:00:00  →  2026-06-30 00:00:00
  actual_pickup_ts       2023-01-01 00:00:00  →  2026-06-30 13:24:00
  planned_delivery_ts    2023-01-03 00:00:00  →  2026-08-08 00:00:00
  actual_delivery_ts     2023-01-01 12:00:00  →  2026-08-09 03:50:24


In [18]:
# Reglas de secuencia lógica en la operación de transporte
r1 = fechas['actual_delivery_ts'] < fechas['actual_pickup_ts']
r2 = fechas['actual_pickup_ts'] < fechas['ship_date']
r3 = fechas['planned_delivery_ts'] < fechas['planned_pickup_ts']

print('=== VIOLACIONES DE SECUENCIA ===')
print(f'  Entrega real anterior al pickup real:   {r1.sum():>5,}')
print(f'  Pickup real anterior al ship_date:      {r2.sum():>5,}')
print(f'  Entrega plan. anterior al pickup plan.: {r3.sum():>5,}')

# Magnitud del desfase en los casos imposibles
if r1.sum() > 0:
    desfase = (fechas.loc[r1, 'actual_pickup_ts'] -
               fechas.loc[r1, 'actual_delivery_ts']).dt.total_seconds() / 3600
    print(f'\n=== DESFASE (horas) en entrega < pickup ===')
    print(desfase.describe().round(1))

    print(f'\n=== Distribución por modo ===')
    print(fechas.loc[r1, 'mode'].value_counts())

=== VIOLACIONES DE SECUENCIA ===
  Entrega real anterior al pickup real:     298
  Pickup real anterior al ship_date:          0
  Entrega plan. anterior al pickup plan.:     0

=== DESFASE (horas) en entrega < pickup ===
count    298.0
mean      48.0
std        0.0
min       48.0
25%       48.0
50%       48.0
75%       48.0
max       48.0
dtype: float64

=== Distribución por modo ===
mode
Parcel     117
LTL         72
TL          54
Ocean       26
Air         17
Drayage     12
Name: count, dtype: int64


In [19]:
casos = fechas[r1].copy()

# Hipótesis: la entrega real es 48h después de lo registrado
casos['entrega_corregida'] = casos['actual_delivery_ts'] + pd.Timedelta(hours=48)

# ¿La corrección produce una secuencia válida?
valida = casos['entrega_corregida'] >= casos['actual_pickup_ts']
print(f'Casos donde +48h produce secuencia válida: {valida.sum()} de {len(casos)}')

# ¿El tránsito resultante es plausible frente al planeado?
casos['transito_corregido'] = ((casos['entrega_corregida'] - casos['actual_pickup_ts'])
                               .dt.total_seconds() / 86400).round(2)

print('\n=== Tránsito corregido vs planeado ===')
print(casos[['planned_transit_days', 'transito_corregido', 'actual_transit_days']].describe().round(2))

print('\n=== Muestra de 5 casos ===')
print(casos[['shipment_id', 'mode', 'actual_pickup_ts', 'actual_delivery_ts',
             'entrega_corregida', 'planned_transit_days', 'actual_transit_days']].head())

Casos donde +48h produce secuencia válida: 298 de 298

=== Tránsito corregido vs planeado ===
       planned_transit_days  transito_corregido  actual_transit_days
count                298.00               298.0               298.00
mean                   7.24                 0.0                 6.58
std                    6.99                 0.0                 7.45
min                    2.00                 0.0                 0.50
25%                    4.00                 0.0                 2.55
50%                    5.00                 0.0                 4.71
75%                    7.00                 0.0                 7.02
max                   42.00                 0.0                42.83

=== Muestra de 5 casos ===
     shipment_id     mode    actual_pickup_ts  actual_delivery_ts   entrega_corregida  planned_transit_days  actual_transit_days
41    SHP1144082       TL 2023-04-20 00:00:00 2023-04-18 00:00:00 2023-04-20 00:00:00                   5.0                 3.37

In [20]:
# Reconstrucción a partir del tránsito registrado, que no fue afectado
casos['entrega_reconstruida'] = (casos['actual_pickup_ts'] +
                                 pd.to_timedelta(casos['actual_transit_days'], unit='D'))

# Validación 1: la secuencia queda correcta
seq_ok = casos['entrega_reconstruida'] > casos['actual_pickup_ts']

# Validación 2: ¿el flag de OTD original es consistente con la fecha reconstruida?
otd_calculado = (casos['entrega_reconstruida'] <= casos['planned_delivery_ts']).astype(int)
coincide = (otd_calculado == casos['on_time_delivery_flag'])

print(f'Secuencia válida tras reconstruir: {seq_ok.sum()} de {len(casos)}')
print(f'OTD calculado coincide con el flag original: {coincide.sum()} de {len(casos)}')

print('\n=== Muestra ===')
print(casos[['shipment_id', 'actual_pickup_ts', 'actual_delivery_ts',
             'entrega_reconstruida', 'actual_transit_days',
             'planned_delivery_ts', 'on_time_delivery_flag']].head())

Secuencia válida tras reconstruir: 298 de 298
OTD calculado coincide con el flag original: 273 de 298

=== Muestra ===
     shipment_id    actual_pickup_ts  actual_delivery_ts entrega_reconstruida  actual_transit_days planned_delivery_ts  on_time_delivery_flag
41    SHP1144082 2023-04-20 00:00:00 2023-04-18 00:00:00  2023-04-23 08:52:48                 3.37          2023-04-25                      1
135   SHP1139470 2023-02-09 00:00:00 2023-02-07 00:00:00  2023-02-17 19:40:48                 8.82          2023-02-19                      1
931   SHP1203370 2024-11-15 00:00:00 2024-11-13 00:00:00  2024-11-17 12:14:24                 2.51          2024-11-19                      1
1282  SHP1137934 2025-12-29 00:00:00 2025-12-27 00:00:00  2025-12-31 08:52:48                 2.37          2026-01-02                      1
3393  SHP1064663 2023-12-21 04:12:00 2023-12-19 04:12:00  2023-12-24 07:33:36                 3.14          2023-12-25                      1


In [21]:
discrepantes = casos[~coincide].copy()

# Distancia entre la entrega reconstruida y el compromiso
discrepantes['margen_horas'] = ((discrepantes['entrega_reconstruida'] -
                                 discrepantes['planned_delivery_ts'])
                                .dt.total_seconds() / 3600).round(2)

print(f'Casos discrepantes: {len(discrepantes)}')
print('\n=== Margen respecto al compromiso (horas) ===')
print(discrepantes['margen_horas'].describe().round(2))

print('\n=== Detalle ===')
print(discrepantes[['shipment_id', 'mode', 'entrega_reconstruida',
                    'planned_delivery_ts', 'margen_horas',
                    'on_time_delivery_flag']].head(15).to_string())

Casos discrepantes: 25

=== Margen respecto al compromiso (horas) ===
count    25.00
mean      5.11
std       3.41
min       0.48
25%       2.46
50%       4.68
75%       8.00
max      11.28
Name: margen_horas, dtype: float64

=== Detalle ===
       shipment_id     mode entrega_reconstruida planned_delivery_ts  margen_horas  on_time_delivery_flag
3635    SHP1059720   Parcel  2023-09-09 05:43:12          2023-09-09          5.72                      1
6495    SHP1021562       TL  2023-11-03 03:00:00          2023-11-03          3.00                      1
8675    SHP1001461  Drayage  2024-09-15 08:24:00          2024-09-15          8.40                      1
14662   SHP1154161      LTL  2024-08-31 06:00:00          2024-08-31          6.00                      1
18876   SHP1029321      LTL  2025-12-16 07:01:12          2025-12-16          7.02                      1
42850   SHP1182335      Air  2024-01-08 06:14:24          2024-01-08          6.24                      1
44359   SHP10313

In [22]:
sano = fechas[~r1 & fechas['actual_delivery_ts'].notna()].copy()

# Criterio A: comparación por timestamp exacto
otd_ts = (sano['actual_delivery_ts'] <= sano['planned_delivery_ts']).astype(int)

# Criterio B: comparación por fecha de calendario
otd_fecha = (sano['actual_delivery_ts'].dt.normalize() <=
             sano['planned_delivery_ts'].dt.normalize()).astype(int)

print(f'Registros evaluados: {len(sano):,}')
print(f'\nCoincidencia con el flag original:')
print(f'  Criterio timestamp exacto: {(otd_ts == sano["on_time_delivery_flag"]).mean()*100:.2f}%')
print(f'  Criterio fecha calendario: {(otd_fecha == sano["on_time_delivery_flag"]).mean()*100:.2f}%')

print(f'\nOTD resultante:')
print(f'  Flag original:      {sano["on_time_delivery_flag"].mean()*100:.2f}%')
print(f'  Criterio timestamp: {otd_ts.mean()*100:.2f}%')
print(f'  Criterio fecha:     {otd_fecha.mean()*100:.2f}%')

Registros evaluados: 217,946

Coincidencia con el flag original:
  Criterio timestamp exacto: 89.87%
  Criterio fecha calendario: 93.49%

OTD resultante:
  Flag original:      83.92%
  Criterio timestamp: 73.80%
  Criterio fecha:     90.43%


### 8.3 Descubrimiento de la regla de OTD

El flag `on_time_delivery_flag` no coincidía con ninguna comparación directa de fechas:

| Criterio | Coincidencia | OTD resultante |
|---|---|---|
| Timestamp exacto | 89.87% | 73.80% |
| Fecha de calendario | 93.49% | 90.43% |
| Flag original | — | **83.92%** |

El valor real quedaba entre ambos extremos, lo que descartó las dos hipótesis iniciales
y sugirió la existencia de una ventana de tolerancia.

**Búsqueda sistemática del parámetro:**

| Tolerancia | Coincidencia | OTD |
|---|---|---|
| 0h | 89.87% | 73.80% |
| 6h | 95.38% | 79.30% |
| **12h** | **100.00%** | **83.92%** |
| 24h | 93.43% | 90.49% |
| 48h | 87.31% | 96.61% |

**Regla identificada:** una entrega se considera a tiempo si ocurre dentro de las
**12 horas posteriores** al compromiso pactado (`planned_delivery_ts`).
Reproduce el flag original con 100% de exactitud.

**Relevancia:** calcular OTD por comparación directa de timestamps —el criterio
intuitivo— arrojaría 73.80%, es decir **10.1 puntos porcentuales por debajo** de la
cifra oficial. Un tablero construido sobre ese supuesto contradiría los reportes del
área de transporte.

Esta regla no estaba documentada: se derivó auditando los casos en que la fecha
reconstruida discrepaba del flag registrado.

In [23]:
# Margen de cada entrega respecto al compromiso, en horas
margen = ((sano['actual_delivery_ts'] - sano['planned_delivery_ts'])
          .dt.total_seconds() / 3600)

print('=== Búsqueda de la tolerancia ===')
print(f'{"Tolerancia":>12} {"Coincidencia":>14} {"OTD result.":>12}')

mejor = (0, 0)
for t in [0, 2, 4, 6, 8, 12, 18, 24, 30, 36, 48]:
    otd_t = (margen <= t).astype(int)
    coin = (otd_t == sano['on_time_delivery_flag']).mean() * 100
    print(f'{t:>10}h {coin:>13.2f}% {otd_t.mean()*100:>11.2f}%')
    if coin > mejor[1]:
        mejor = (t, coin)

print(f'\nMejor tolerancia: {mejor[0]}h con {mejor[1]:.2f}% de coincidencia')
print(f'OTD reportado por el flag: {sano["on_time_delivery_flag"].mean()*100:.2f}%')

=== Búsqueda de la tolerancia ===
  Tolerancia   Coincidencia  OTD result.
         0h         89.87%       73.80%
         2h         91.77%       75.70%
         4h         93.57%       77.49%
         6h         95.38%       79.30%
         8h         96.99%       80.91%
        12h        100.00%       83.92%
        18h         96.34%       87.58%
        24h         93.43%       90.49%
        30h         91.21%       92.71%
        36h         89.56%       94.36%
        48h         87.31%       96.61%

Mejor tolerancia: 12h con 100.00% de coincidencia
OTD reportado por el flag: 83.92%


In [24]:
fechas.to_pickle('../data/processed/01_fechas_convertidas.pkl')
print(f'Guardado: {len(fechas):,} filas')

Guardado: 220,000 filas


---

# 9. Conclusiones del diagnóstico

## 9.1 Estado del archivo evaluado

| Métrica | Valor |
|---|---|
| Registros en la extracción cruda | 221,320 |
| Registros únicos válidos | 220,000 |
| Columnas | 39 |
| Periodo cubierto | 2023-01-01 → 2026-06-30 |
| Columnas con valores faltantes | 6 (máximo 3.0%) |

## 9.2 Hallazgos

**1. Duplicados enmascarados por inconsistencia de formato**

De 1,320 registros duplicados, solo 1,186 eran detectables como duplicados exactos.
Los 134 restantes diferían únicamente en formato (`Mexico` vs `MX`, `Truckload` vs `TL`).

*Consecuencia:* la normalización de categóricas debe preceder a la deduplicación.
El orden inverso —el más intuitivo— deja 134 registros duplicados en el dataset,
inflando el conteo de embarques y el gasto total.

**2. Corrupción determinista, no aleatoria**

Once registros duplicados presentaban una copia sana y una corrupta, funcionando como
muestra de validación natural. Confirmaron que la corrupción responde a transformaciones
fijas: `total_cost_usd × (−1)` y `weight_kg × 1000`.

*Consecuencia:* las reglas de corrección aplicadas a los ~640 casos sin gemelo se
sustentan en evidencia observada, no en supuestos.

**3. Los umbrales de detección deben ser relativos al modo de transporte**

Un umbral global identificó 159 pesos inflados; el criterio por modo identificó 242.
La diferencia son principalmente envíos Parcel: 3 kg inflados resultan en 3,000 kg,
absurdo para paquetería pero indetectable frente a un umbral construido sobre TL y Ocean,
cuyas medianas son tres órdenes de magnitud mayores.

*Consecuencia:* en operaciones multimodales, cualquier detección de outliers debe
segmentarse por modo.

**4. Fechas corrompidas reconstruibles**

298 registros presentaban entrega anterior al pickup, todos con desfase de exactamente
48.0 horas (desviación estándar = 0). El valor original se recupera desde
`actual_transit_days`, columna que no fue afectada por la corrupción.

*Consecuencia:* se preservan 298 mediciones de servicio que un criterio de descarte
habría eliminado.

**5. Regla de negocio no documentada: tolerancia de 12 horas en OTD**

El flag `on_time_delivery_flag` no coincidía con ninguna comparación directa de fechas.
La búsqueda sistemática identificó una ventana de gracia de 12 horas posteriores al
compromiso, que reproduce el flag con 100% de exactitud.

*Consecuencia:* calcular OTD por comparación directa arrojaría 73.80% frente al 83.92%
oficial —10.1 puntos de diferencia—, generando reportes que contradirían al sistema fuente.

## 9.3 Especificación del pipeline de limpieza

El orden de ejecución no es intercambiable: los pasos 1 y 2 deben mantener esta
secuencia, y el paso 4 depende de la conversión temporal del paso 3.

| # | Paso | Regla derivada del diagnóstico | Registros afectados |
|---|---|---|---|
| 1 | Normalizar categóricas | Diccionario explícito de sinónimos; `strip()` + `upper()` es insuficiente | ~11,100 |
| 2 | Deduplicar | Conservar la versión que satisface más reglas de validez, no `keep='first'` | 1,320 |
| 3 | Convertir tipos temporales | `errors='coerce'` sobre las 5 columnas de fecha | 220,000 |
| 4 | Reconstruir fechas imposibles | `actual_pickup_ts + actual_transit_days` | 298 |
| 5 | Corregir costos negativos | Invertir signo; validar contra la suma de componentes | 393 |
| 6 | Corregir pesos inflados | División entre 1000 con criterio de ida y vuelta por modo | 242 |
| 7 | Tratar nulos | Criterio diferenciado según uso analítico de cada columna | ~19,700 |
| 8 | Derivar métricas | OTD con tolerancia de 12h; costo por kg y por km | 220,000 |

### Criterio de tratamiento de nulos

| Columna | Nulos | Tratamiento | Justificación |
|---|---|---|---|
| `carrier_id` | 887 | Descartar registro | Sin transportista el registro no es utilizable para el scorecard, que es el eje del análisis |
| `incoterm` | 6,640 | Etiquetar `NO_APLICA` | La ausencia es informativa: el incoterm no aplica en tráfico doméstico |
| `accessorial_usd` | 3,318 | Imputar 0 | La ausencia de cargo accesorial equivale a cargo cero |
| `weight_kg` | 2,648 | Mediana por modo | Las escalas difieren por modo; la mediana resiste el sesgo de la distribución |
| `volume_cbm` | 4,422 | Mediana por modo | Mismo criterio |
| `actual_delivery_ts` | 1,767 | Conservar como nulo | Puede corresponder a embarques en tránsito; imputarla falsearía el cálculo de OTD |

    ## 9.4 Limitaciones

- **Cobertura de pesos inflados:** se detectan 242 de los ~250 estimados. Los restantes
  presentan valores que, aun inflados, caen dentro del rango legítimo de su modo,
  resultando estadísticamente indistinguibles de mediciones reales. Se documenta la
  limitación en lugar de relajar el criterio para forzar la cifra esperada.

- **Nulos en `actual_delivery_ts`:** no es posible distinguir entre embarques
  efectivamente en tránsito y pérdidas de dato en la extracción. El análisis de OTD
  se calcula sobre los registros con entrega confirmada, declarando el denominador.

- **Naturaleza de los datos:** dataset sintético generado para replicar la estructura
  de un TMS. Los hallazgos demuestran la metodología de auditoría, no condiciones
  operativas de una empresa real.

## 9.5 Siguiente etapa

Implementación del pipeline especificado en 9.2 como módulo reutilizable
(`src/limpieza.py`), con validación del resultado contra `fact_shipments_clean.csv`
y generación del dataset analítico para las fases de análisis y visualización.